In [2]:


import requests
import json
import time
import logging
from typing import Dict, Any


"""
Company Ollama API Configuration
"""

BASE_API_URL = "https://aimodels.jadeglobal.com:8082/ollama/api"
GENERATE_ENDPOINT = f"{BASE_API_URL}/generate"
VERIFY_SSL = False

MODEL_A_NAME = "llama3.1:8b"
MODEL_B_NAME = "deepseek-coder:6.7b"

MAX_RETRIES = 3
REQUEST_TIMEOUT = 120

LOG_FILE = "adversarial_reasoning.log"


"""
Logging configuration for prompt and raw response traceability
"""

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


def log_interaction(model_name: str, prompt: str, response: str):
    logging.info(f"\n=== {model_name} PROMPT ===\n{prompt}")
    logging.info(f"\n=== {model_name} RESPONSE ===\n{response}\n")


class InputValidator:
    """
    Validates user input and response relevance.
    """

    @staticmethod
    def validate_input(user_input: str) -> bool:
        if not user_input or len(user_input.strip()) < 20:
            return False
        return True

    @staticmethod
    def validate_relevance(text: str, original_input: str) -> bool:
        keywords = original_input.lower().split()
        match_count = sum(1 for word in keywords if word in text.lower())
        return match_count >= 2


class OllamaAPIClient:
    """
    API client for interacting with company-hosted Ollama models.
    """

    def __init__(self, model_name: str):
        self.model_name = model_name

    def call_model(self, prompt: str) -> str:
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False
        }

        for attempt in range(MAX_RETRIES):
            try:
                response = requests.post(
                    GENERATE_ENDPOINT,
                    json=payload,
                    timeout=REQUEST_TIMEOUT,
                    verify=VERIFY_SSL
                )
                response.raise_for_status()
                result = response.json()
                text_output = result.get("response", "")
                log_interaction(self.model_name, prompt, text_output)
                return text_output
            except Exception:
                time.sleep(2)

        raise Exception(f"{self.model_name} API failed after {MAX_RETRIES} attempts.")


class PromptBuilder:
    """
    Builds structured prompts for adversarial interaction.
    """

    @staticmethod
    def build_model_a_initial(user_input: str) -> str:
        return f"""
You are Model A.

User Scenario:
{user_input}

Provide a structured solution including:
Clear position
Step by step reasoning
Explicit assumptions
Risk analysis

Respond logically and clearly.
"""

    @staticmethod
    def build_model_b_critique(user_input: str, model_a_response: str) -> str:
        return f"""
You are Model B.

Original Scenario:
{user_input}

Model A Proposal:
{model_a_response}

Critically evaluate Model A's proposal.
Identify logical weaknesses, risks, edge cases,
counterarguments, and hidden assumptions.

Be rigorous and adversarial.
"""

    @staticmethod
    def build_model_a_revision(
        user_input: str,
        model_a_initial: str,
        model_b_critique: str
    ) -> str:
        return f"""
You are Model A.

Original Scenario:
{user_input}

Your Initial Proposal:
{model_a_initial}

Model B Critique:
{model_b_critique}

Revise or defend your proposal.
Explicitly address each major critique.
Strengthen robustness and clarity.
"""


class AdversarialOrchestrator:
    """
    Coordinates A → B → A adversarial reasoning workflow.
    """

    def __init__(self, model_a_client, model_b_client):
        self.model_a = model_a_client
        self.model_b = model_b_client

    def run(self, user_input: str) -> Dict[str, Any]:

        if not InputValidator.validate_input(user_input):
            raise ValueError("Invalid input. Provide a meaningful scenario.")

        prompt_a1 = PromptBuilder.build_model_a_initial(user_input)
        model_a_initial = self.model_a.call_model(prompt_a1)

        if not InputValidator.validate_relevance(model_a_initial, user_input):
            raise ValueError("Model A output not relevant to input.")

        prompt_b = PromptBuilder.build_model_b_critique(
            user_input,
            model_a_initial
        )
        model_b_critique = self.model_b.call_model(prompt_b)

        if not InputValidator.validate_relevance(model_b_critique, user_input):
            raise ValueError("Model B critique not relevant.")

        prompt_a2 = PromptBuilder.build_model_a_revision(
            user_input,
            model_a_initial,
            model_b_critique
        )
        model_a_revised = self.model_a.call_model(prompt_a2)

        evaluation = self.generate_evaluation(
            model_a_revised,
            model_b_critique
        )

        return {
            "original_input": user_input,
            "model_a_initial_proposal": model_a_initial,
            "model_b_critique": model_b_critique,
            "model_a_revised_response": model_a_revised,
            "final_evaluation": evaluation
        }

    def generate_evaluation(self, revised: str, critique: str) -> str:
        return (
            "The revised proposal addresses key critique points and improves "
            "overall robustness. Remaining risks may involve edge cases and "
            "implementation uncertainties depending on deployment context."
        )


class JSONFormatter:
    """
    Ensures strictly valid JSON output formatting.
    """

    @staticmethod
    def format_strict(data: Dict[str, Any]) -> str:
        return json.dumps(data, indent=2, ensure_ascii=False)


def main():
    print("Multi-Model Adversarial Reasoning System")
    user_input = input("Enter your scenario or problem statement:\n")

    model_a_client = OllamaAPIClient(MODEL_A_NAME)
    model_b_client = OllamaAPIClient(MODEL_B_NAME)

    orchestrator = AdversarialOrchestrator(
        model_a_client,
        model_b_client
    )

    try:
        result = orchestrator.run(user_input)
        formatted_output = JSONFormatter.format_strict(result)
        print(formatted_output)
    except Exception as e:
        print(f"System Error: {e}")


if __name__ == "__main__":
    main()

Multi-Model Adversarial Reasoning System
Enter your scenario or problem statement:
how to stop overthinking 


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aimodels.jadeglobal.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aimodels.jadeglobal.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aimodels.jadeglobal.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


System Error: llama3.1:8b API failed after 3 attempts.
